# Student Performance — EDA, Unsupervised und Supervised Learning

## Schritt 1: Werkzeuge laden

Wir verwenden folgende Werkzeuge:

* `read_csv` aus `pandas`
* `train_test_split` aus dem `model_selection`-Modul in `scikit-learn`
* `KMeans` aus dem `cluster`-Modul in `scikit-learn` (für Unsupervised Learning)
* `DecisionTreeRegressor` aus dem `tree`-Modul in `scikit-learn`
* `LinearRegression` aus dem `linear_model`-Modul in `scikit-learn`
* `PolynomialFeatures` aus dem `preprocessing`-Modul in `scikit-learn`
* `r2_score` aus dem `metrics`-Modul in `scikit-learn`
* `mean_absolute_error` aus dem `metrics`-Modul in `scikit-learn`
* `root_mean_squared_error` aus dem `metrics`-Modul in `scikit-learn`

In [ ]:
# daten einlesen
from pandas import read_csv, DataFrame

# train/test-split
from sklearn.model_selection import train_test_split

# explorieren und visualisieren
from seaborn import pairplot, heatmap
from matplotlib import pyplot as plt

# unsupervised-modell
from sklearn.cluster import KMeans

# supervised-modelle
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression

# preprocessing-pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# polynomial-features für die polynomialregression
from sklearn.preprocessing import PolynomialFeatures

# metriken
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

# NaN-platzhalter für die bereinigung
from numpy import nan

## Schritt 2: Daten Laden

Wir laden den `student_performance_data.csv`-Datensatz. Die CSV hat zwei Eigenheiten, die wir hier berücksichtigen:

* Zeile 0 ist eine Beschreibung (kein Header) — wir überspringen sie.
* Hinter den 33 echten Spalten gibt es 21 leere Trailing-Spalten — wir laden nur die ersten 33.

Zusätzlich werfen wir nur komplett leere Zeilen raus. Einzelne `NaN`-Werte behalten wir und füllen sie später in der Pipeline mit `SimpleImputer` auf.

In [ ]:
# csv laden: semikolon als trenner, beschreibungszeile (zeile 0) überspringen, 33 echte spalten
data = read_csv("../data/student_performance_data.csv", sep=";", skiprows=[0], usecols=range(33))

# nur komplett leere zeilen raus — einzelne NaN behalten wir für den imputer
data = data.dropna(how="all")

## Schritt 3: Daten explorieren (EDA)

Wir schauen uns die Daten an: Statistiken, einzigartige Werte, Bereiche und Plots. Ziel ist eine Liste aller Auffälligkeiten, die wir in Schritt 4 bereinigen.

In [ ]:
# erster blick auf die daten
data.head()

In [ ]:
# datentypen, füllstand, statistische kennzahlen
data.info()
data.describe()

### Kategoriale Spalten — welche Werte kommen vor?

Wir prüfen, ob in den kategorialen Spalten nur die in der [README](../data/student_performance_data.md) dokumentierten Werte vorkommen.

In [ ]:
# einzigartige werte pro kategoriale spalte
for col in data.select_dtypes(include="object").columns:
    print(f"{col}: {sorted(data[col].dropna().unique().tolist())}")

### Numerische Spalten — Werte außerhalb der dokumentierten Bereiche?

Wir vergleichen jede numerische Spalte gegen den in der README definierten gültigen Bereich (z.B. `age` zwischen 15 und 22).

In [ ]:
# erlaubte bereiche laut readme
valid_ranges = {
    "age": (15, 22),
    "Medu": (0, 4), "Fedu": (0, 4),
    "traveltime": (1, 4), "studytime": (1, 4),
    "failures": (0, 4),
    "famrel": (1, 5), "freetime": (1, 5), "goout": (1, 5),
    "Dalc": (1, 5), "Walc": (1, 5), "health": (1, 5),
    "absences": (0, 93),
    "G1": (0, 20), "G2": (0, 20), "G3": (0, 20),
}

# werte außerhalb der bereiche finden
for col, (lo, hi) in valid_ranges.items():
    bad = data[(data[col] < lo) | (data[col] > hi)]
    if len(bad) > 0:
        print(f"{col}: {len(bad)} werte außerhalb [{lo}, {hi}] -> {bad[col].tolist()}")

### Visualisierungen

In [ ]:
# verteilung der periodennoten
data[["G1", "G2", "G3"]].plot(kind="box")
plt.title("verteilung der noten G1, G2, G3")

In [ ]:
# age-boxplot macht die outlier sichtbar
data[["age"]].plot(kind="box")
plt.title("verteilung von age")
plt.savefig("../img/age_distribution.png")

In [ ]:
# häufigkeit der schulen (GP vs MS)
data["school"].value_counts().plot(kind="barh")
plt.title("häufigkeit school")
plt.xlabel("Anzahl Schüler")
plt.ylabel("Schule")

In [ ]:
# häufigkeit der berufe der mutter
data["Mjob"].value_counts().plot(kind="barh")
plt.title("häufigkeit Mjob")
plt.xlabel("Anzahl Schüler")
plt.ylabel("Beruf der Mutter")

In [ ]:
# häufigkeit der gründe für die schulwahl
data["reason"].value_counts().plot(kind="barh")
plt.title("häufigkeit reason")
plt.xlabel("Anzahl Schüler")
plt.ylabel("Grund für Schulwahl")

In [ ]:
# verteilung der endnote G3 (zielvariable)
data["G3"].plot(kind="hist", bins=20)
plt.title("verteilung G3")
plt.xlabel("Endnote G3")
plt.ylabel("Anzahl Schüler")

In [ ]:
# verteilung der fehlzeiten (sehr schief)
data["absences"].plot(kind="hist", bins=30)
plt.title("verteilung absences")
plt.xlabel("Anzahl Fehlzeiten")
plt.ylabel("Anzahl Schüler")

In [ ]:
# korrelationsmatrix der numerischen spalten
heatmap(data.select_dtypes(include="number").corr())
plt.title("korrelationsmatrix der numerischen spalten")

In [ ]:
# zusammenhang zwischen relevanten features und G3
pairplot(data[["studytime", "failures", "absences", "G1", "G2", "G3"]])

### Zusammenfassung der Funde

Diese Probleme bereinigen wir in Schritt 4:

1. **`sex` enthält `'Female'`** neben `'F'` und `'M'` — wir normalisieren auf `'F'`/`'M'`.
2. **`age` hat 8 unplausible Werte** (z.B. `1511`, `11237`, `999`) weit außerhalb des erlaubten Bereichs `[15, 22]` — wir setzen sie auf `NaN`.
3. **Fehlende Werte:** Schritt 2 lässt mit `dropna(how="all")` jetzt nur komplett leere Zeilen raus. Die 13 Zeilen mit einzelnen `NaN`s bleiben drin und werden in der Pipeline durch `SimpleImputer` aufgefüllt.
4. **`G1`, `G2` als starke Prädiktoren für `G3`** — die Korrelation ist sehr hoch (siehe Heatmap). Laut [README](../data/student_performance_data.md) ist `G3` *ohne* `G1`/`G2` die interessantere Vorhersage (Frühwarnsystem). Diese Variante modellieren wir.
5. **`float64` statt `int` bei Ganzzahl-Spalten** — entsteht durch die `NaN`s. Funktional kein Problem fürs ML, lassen wir so.

## Schritt 4: Daten berichtigen

Wir bereinigen die in Schritt 3 gefundenen Probleme: `'Female'` → `'F'` normalisieren und `age`-Outlier auf `NaN` setzen. Den Erfolg prüfen wir direkt mit einem Boxplot.

In [ ]:
# 'Female' -> 'F'
data["sex"] = data["sex"].replace("Female", "F")

# age-werte außerhalb [15, 22] auf NaN — der imputer füllt sie später auf
data.loc[(data["age"] < 15) | (data["age"] > 22), "age"] = nan

In [ ]:
# gegencheck: age nach der bereinigung — outlier sind weg
data[["age"]].plot(kind="box")
plt.title("verteilung von age nach bereinigung")

## Schritt 5: Daten vorbereiten

Wir wählen Features und Target, teilen in Trainings- und Test-Set und bauen die Preprocessing-Pipeline (Imputer + Skalierung + One-Hot-Encoding).

In [ ]:
# features: alles außer den noten
# G1/G2 lassen wir bewusst weg -> frühwarnsystem-variante (siehe readme)
X = data.drop(columns=["G1", "G2", "G3"])

# target: die endnote
y = data["G3"]

### Aufteilung in Trainings- und Test-Set

In [ ]:
# in trainings- und test-set aufteilen
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# spalten automatisch nach typ trennen
num_selector = make_column_selector(dtype_include="number")
cat_selector = make_column_selector(dtype_exclude="number")

# numerisch: NaN -> median, dann skalieren
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler()),
])

# kategorial: NaN -> 'missing', dann one-hot
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_transformer, num_selector),
    ("cat", categorical_transformer, cat_selector),
])

In [ ]:
# pipeline nur auf den trainingsdaten fitten, dann beide sets transformieren
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

print("Trainings-Daten vorbereitet:", X_train_prep.shape)
print("Test-Daten vorbereitet:    ", X_test_prep.shape)

## Schritt 6: Unsupervised Learning

Wir verwenden `KMeans` (wie in [02_unsupervised_learning.ipynb](../../data-science-ue-ct/notebooks/02_unsupervised_learning.ipynb)), um natürliche Gruppen von Schülern in den vorbereiteten Trainingsdaten zu finden. Danach prüfen wir, ob die Endnote `G3` zwischen den Gruppen unterschiedlich ist — wenn ja, hat das Clustering eine sinnvolle Struktur entdeckt.

In [ ]:
# K-Means mit k=3 trainieren und cluster-zuordnung für die trainingsdaten holen
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters_train = kmeans.fit_predict(X_train_prep)

In [ ]:
# wie unterscheidet sich die endnote G3 zwischen den clustern?
cluster_summary = DataFrame({"cluster": clusters_train, "G3": y_train.values}).groupby("cluster")["G3"].agg(["mean", "std", "count"])
print(cluster_summary)

## Schritt 7: Supervised Learning

Wir trainieren zwei Modelle (`DecisionTreeRegressor` und `LinearRegression`), erzeugen Vorhersagen für die Testdaten und bewerten die Qualität mit `r2_score`, `mean_absolute_error` und `root_mean_squared_error`. Anschließend probieren wir verschiedene `max_depth`-Werte beim Decision Tree aus, um den kleinsten Fehler zu finden.

In [ ]:
# decision tree mit den besten hyperparametern (siehe tuning-zelle unten)
#
# probiert wurden alle kombinationen aus:
#   max_depth        ∈ {2, 3, 5, 7, 10, None}
#   min_samples_leaf ∈ {1, 5, 10, 15, 20}
#
# bestes ergebnis: max_depth=3, min_samples_leaf=10  -> R2 ≈ 0.153
tree_regressor = DecisionTreeRegressor(max_depth=3, min_samples_leaf=10, random_state=42)
tree_regressor.fit(X_train_prep, y_train)

# vorhersagen auf den testdaten
y_pred_tree = tree_regressor.predict(X_test_prep)

# qualität bewerten
print("Decision Tree (beste hyperparameter)")
print("R2  ", r2_score(y_test, y_pred_tree))
print("MAE ", mean_absolute_error(y_test, y_pred_tree))
print("RMSE", root_mean_squared_error(y_test, y_pred_tree))

In [ ]:
# lineare regression
linear_regressor = LinearRegression()
linear_regressor.fit(X_train_prep, y_train)

# vorhersagen auf den testdaten
y_pred_linear = linear_regressor.predict(X_test_prep)

# qualität bewerten
print("Lineare Regression")
print("R2  ", r2_score(y_test, y_pred_linear))
print("MAE ", mean_absolute_error(y_test, y_pred_linear))
print("RMSE", root_mean_squared_error(y_test, y_pred_linear))

In [ ]:
# overfitting-check: train- vs test-R2 vergleichen
# liegen die nah beieinander -> kein overfitting
print("Decision Tree      R2 Training:", r2_score(y_train, tree_regressor.predict(X_train_prep)))
print("Decision Tree      R2 Test:    ", r2_score(y_test, y_pred_tree))
print("Lineare Regression R2 Training:", r2_score(y_train, linear_regressor.predict(X_train_prep)))
print("Lineare Regression R2 Test:    ", r2_score(y_test, y_pred_linear))

In [ ]:
# die echten testwerte (zum vergleich mit den vorhersagen)
y_test.values.ravel()

### Hyperparameter-Tuning (Decision Tree)

Wir testen ein Grid aus `max_depth` und `min_samples_leaf` und identifizieren die beste Kombination (höchster R²-Wert). Genau dieses Ergebnis verwenden wir oben im Haupt-Decision-Tree.

In [ ]:
# alle kombinationen testen, beste kombination finden und ausgeben
print("Decision Tree — Hyperparameter-Grid:")
print(f"{'max_depth':>10} {'min_leaf':>10} {'R2':>8} {'MAE':>8} {'RMSE':>8}")

best_params, best_r2 = None, -float("inf")
for d in [2, 3, 5, 7, 10, None]:
    for leaf in [1, 5, 10, 15, 20]:
        t = DecisionTreeRegressor(max_depth=d, min_samples_leaf=leaf, random_state=42)
        t.fit(X_train_prep, y_train)
        p = t.predict(X_test_prep)
        r2 = r2_score(y_test, p)
        mae = mean_absolute_error(y_test, p)
        rmse = root_mean_squared_error(y_test, p)
        print(f"{str(d):>10} {leaf:>10} {r2:>8.3f} {mae:>8.3f} {rmse:>8.3f}")
        if r2 > best_r2:
            best_params, best_r2 = (d, leaf), r2

print(f"\nBeste Kombination: max_depth={best_params[0]}, min_samples_leaf={best_params[1]}  -> R2={best_r2:.3f}")

## Extra-Schritt: Polynomialregression

Wir heben die `LinearRegression` mit `PolynomialFeatures` auf ein höheres Polynomiallevel und vergleichen sie mit der normalen Linearen. Danach probieren wir verschiedene Grade aus.

In [ ]:
# polynomial-features grad 2 auf die vorbereiteten daten anwenden
poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train_prep)
X_test_poly = poly.transform(X_test_prep)

# lineare regression auf den polynomial-features
poly_regressor = LinearRegression()
poly_regressor.fit(X_train_poly, y_train)
y_pred_poly = poly_regressor.predict(X_test_poly)

print("Polynomialregression (Grad 2)")
print("R2  ", r2_score(y_test, y_pred_poly))
print("MAE ", mean_absolute_error(y_test, y_pred_poly))
print("RMSE", root_mean_squared_error(y_test, y_pred_poly))

In [ ]:
# vergleich: normale lineare regression vs. polynomialregression
print("Linear normal       R2 Test: ", r2_score(y_test, y_pred_linear))
print("Polynomial (Grad 2) R2 Test: ", r2_score(y_test, y_pred_poly))
print("Polynomial (Grad 2) R2 Train:", r2_score(y_train, poly_regressor.predict(X_train_poly)))

### Polynomial-Grad-Tuning

Wir probieren verschiedene Polynomial-Grade aus (1 = wie normale Lineare, 2 = quadratisch, 3 = kubisch).

In [ ]:
# polynomialregression mit verschiedenen graden testen
print("Polynomialregression — verschiedene grade:")
for deg in [1, 2, 3]:
    pf = PolynomialFeatures(degree=deg)
    xtr = pf.fit_transform(X_train_prep)
    xte = pf.transform(X_test_prep)
    m = LinearRegression()
    m.fit(xtr, y_train)
    p = m.predict(xte)
    print(f"  degree={deg}  R2={r2_score(y_test, p):.3f}  MAE={mean_absolute_error(y_test, p):.3f}  RMSE={root_mean_squared_error(y_test, p):.3f}")

## Gesamtvergleich der Modelle

Zusammenfassung aller drei Basis-Modelle (Decision Tree, Lineare Regression, Polynomialregression Grad 2) auf dem Testset.

In [ ]:
# zusammenfassung: vergleich der drei basis-modelle auf dem testset
print(f"{'Modell':<28}{'R2':>8}{'MAE':>8}{'RMSE':>8}")
print(f"{'Decision Tree':<28}{r2_score(y_test, y_pred_tree):>8.3f}{mean_absolute_error(y_test, y_pred_tree):>8.3f}{root_mean_squared_error(y_test, y_pred_tree):>8.3f}")
print(f"{'Lineare Regression':<28}{r2_score(y_test, y_pred_linear):>8.3f}{mean_absolute_error(y_test, y_pred_linear):>8.3f}{root_mean_squared_error(y_test, y_pred_linear):>8.3f}")
print(f"{'Polynomial (Grad 2)':<28}{r2_score(y_test, y_pred_poly):>8.3f}{mean_absolute_error(y_test, y_pred_poly):>8.3f}{root_mean_squared_error(y_test, y_pred_poly):>8.3f}")